# Unsupervised Models

In [1]:
import pandas as pd

#### Import preprocessed data

In [2]:
PROCESSED_DATA_PATH = 'data/processed'

# 1 - non-scaled

X_train_non_scaled  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_non_scaled.csv')
X_val_non_scaled    = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_non_scaled.csv')
X_test_non_scaled   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_non_scaled.csv')

# 2 - robust scaled

X_train_scaled_robust = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_robust_scaled.csv')
X_val_scaled_robust   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_robust_scaled.csv')
X_test_scaled_robust  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_robust_scaled.csv')

# 3 - min-max scaled

X_train_scaled_minmax = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_minmax_scaled.csv')
X_val_scaled_minmax   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_minmax_scaled.csv')
X_test_scaled_minmax  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_minmax_scaled.csv')

# 4 - standard scaled
X_train_scaled_standard = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_standard_scaled.csv')
X_val_scaled_standard   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_standard_scaled.csv')
X_test_scaled_standard  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_standard_scaled.csv')

# y
y_train = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_train.csv')
y_val   = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_val.csv')


X_train_full = pd.concat([X_train_non_scaled, X_val_non_scaled])
y_train_full = pd.concat([y_train, y_val])

print("\nAll versions loaded.")


All versions loaded.


In [3]:
'''
training.py
'''
import json
from datetime import datetime
import os
import shutil

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.utils.parallel import Parallel, delayed
import joblib
import wandb
import matplotlib.pyplot as plt


def refit_full(model, X_full: pd.DataFrame, y_full: pd.Series) -> None:
    """Refit best model on train+val combined before submission."""

    print("Refitting best model on full training data (train + val)...")
    model.fit(X_full, y_full)


def compute_metrics(y_real: pd.Series, y_pred: pd.Series) -> list[float]:
    accuracy = accuracy_score(y_real,y_pred)
    f1_macro =f1_score(y_real,y_pred, average='macro')
    precision_macro =precision_score(y_real,y_pred,  average='macro')
    recall_macro =recall_score(y_real,y_pred,  average='macro')
    classif_report = classification_report(y_real, y_pred)
    return [accuracy, f1_macro, precision_macro, recall_macro, classif_report]


def evaluate(base_name: str, model, X_val: pd.DataFrame, y_val: pd.Series,  best_params: dict|None = None) -> None:
    '''
    Evaluates the best model on the validation data and defines the experiment.
    '''
    print("Evaluating best model on unseen validation data...")

    y_pred = model.predict(X_val)

    define_experiment(base_name, compute_metrics(y_val, y_pred), y_val, y_pred)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics import confusion_matrix

def plot_spatial_confusion(y_true, y_pred, base_name="Model"):
    # 1. Define physical coordinates (radius, angle_in_degrees) based on your image
    polar_coords = {
        0: (2, 0),   1: (2, 45),  2: (2, 90),  3: (2, 135), 4: (2, 180),
        5: (5, 0),   6: (5, 45),  7: (5, 90),  8: (5, 135), 9: (5, 180)
    }

    # Convert polar to Cartesian (x, y) coordinates for plotting
    coords = {}
    for label, (r, theta) in polar_coords.items():
        rad = np.radians(theta)
        coords[label] = (r * np.cos(rad), r * np.sin(rad))

    # Calculate standard confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(10))

    fig, ax = plt.subplots(figsize=(12, 8))

    # 2. Draw the Tracking Unit (centered at origin, pointing outward)
    tracking_unit = patches.Rectangle((-1.5, -1.5), 3, 1.5, color='black', zorder=5)
    ax.add_patch(tracking_unit)
    ax.text(0, -0.75, 'Tracking\nunit', color='white', ha='center', va='center', weight='bold', zorder=6)

    # 3. Draw the background dashed guidelines
    for label in [5, 6, 7, 8, 9]:
        x, y = coords[label]
        ax.plot([0, x], [0, y], color='black', linestyle='--', alpha=0.6, zorder=1)

    # 4. Plot the position nodes (0 through 9)
    for label, (x, y) in coords.items():
        ax.plot(x, y, 'o', markersize=25, color='white', markeredgecolor='black', zorder=4)
        ax.text(x, y, str(label), ha='center', va='center', fontsize=12, zorder=5)

    # 5. Draw the confusion arrows
    # Find the maximum off-diagonal value so we can scale the arrow thickness
    off_diag_mask = ~np.eye(cm.shape[0], dtype=bool)
    max_conf = np.max(cm[off_diag_mask]) if np.any(cm[off_diag_mask]) else 1

    for i in range(10):
        for j in range(10):
            # Only plot off-diagonal elements (errors) where count > 0
            if i != j and cm[i, j] > 0:
                count = cm[i, j]
                x1, y1 = coords[i] # True position
                x2, y2 = coords[j] # Predicted position (where it was mistakenly placed)

                # Scale arrow thickness (linewidth) and opacity (alpha) based on error frequency
                lw = max(1, (count / max_conf) * 5)
                alpha = min(0.3 + (count / max_conf) * 0.7, 1.0)

                # Create a curved arrow so bidirectional confusions (i->j and j->i) don't overlap
                arrow = patches.FancyArrowPatch(
                    (x1, y1), (x2, y2),
                    connectionstyle="arc3,rad=0.15",
                    arrowstyle="->,head_length=8,head_width=4",
                    color='red',
                    linewidth=lw,
                    alpha=alpha,
                    shrinkA=15, # Leaves a gap so the arrow doesn't overlap the circle text
                    shrinkB=15,
                    zorder=3
                )
                ax.add_patch(arrow)

    # 6. Formatting to match the physical aspect ratio

    ax.set_aspect('equal')
    ax.set_xlim(-6, 6)
    ax.set_ylim(-2, 6)
    ax.axis('off')
    plt.title(f'Spatial Error Map: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()

    return fig


def define_experiment(base_name: str, metrics: list[float], y_val: pd.Series, y_pred: pd.Series,  best_params: dict|None = None) -> None:

    accuracy, f1_macro, precision_macro, recall_macro, classif_report = metrics
    experiment_results = {
        "model_name": base_name,
        "best_hyperparameters": best_params,
        "validation_metrics": {"accuracy": accuracy, "f1_macro": f1_macro}
    }

    print("Initializing Weights & Biases run...")
    wandb.init(
        project="AA1",
        entity="laura-rebollo-crespo-universitat-polit-cnica-de-catalunya",
        name=f"{base_name}_f1-{f1_macro:.4f}",
        config={"model_name": base_name, "best_params": best_params})

    fig, ax = plt.subplots(figsize=(10, 8))

    disp = ConfusionMatrixDisplay.from_predictions(
        y_val,
        y_pred,
        ax=ax,
        cmap='viridis',
        colorbar=False
    )
    plt.title(f'Confusion Matrix: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()
    spatial = plot_spatial_confusion(y_val, y_pred, base_name)

    # LOG IT TO W&B
    wandb.log({
        "val_accuracy": accuracy,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "classification_report": wandb.Html(f"<pre>{classif_report}</pre>"),

        "scikit_learn_matrix": wandb.Image(fig),
        "spatial_confusion_matrix": wandb.Image(spatial)
    })


    plt.close(fig)
    plt.close(spatial)


def save(base_name:str, model) -> None:
    """
    Saves the best model locally and uploads it to W&B as an artifact, then cleans up the local file.
    """
    models_dir = "outputs/models"
    os.makedirs(models_dir, exist_ok=True)

    # Save the Model locally first so W&B can grab it
    model_filepath = f"{models_dir}/{base_name}.pkl"
    print(f"Saving model locally to {model_filepath}...")
    joblib.dump(model, model_filepath)

    # --- 3. UPLOAD MODEL TO W&B ---
    print("Uploading model to W&B Cloud...")
    model_artifact = wandb.Artifact(
        name=f"{base_name}_model",
        type="model",
        description="Trained  model"
    )
    model_artifact.add_file(model_filepath)
    wandb.log_artifact(model_artifact)

    if os.path.exists(model_filepath):
        os.remove(model_filepath)
        # Only remove directory if it's empty; use shutil.rmtree if cleanup needed
        try:
            os.rmdir(models_dir)
        except OSError:
            pass  # Directory not empty or other error - that's fine
        print("Deleted:", model_filepath)
    else:
        print("File not found:", model_filepath)


def save_submission(y_pred: pd.Series, file_name: str) -> None:
    """
    Generates Kaggle predictions and uploads the CSV to W&B.
    """
    submissions_dir = "outputs/submissions"
    os.makedirs(submissions_dir, exist_ok=True)

    output_path = f"{submissions_dir}/{file_name}.csv"

    print("Generating Kaggle submission...")

    submission_ids = range(len(y_pred))

    submission = pd.DataFrame({
        "ID": submission_ids,
        "POSITION": y_pred.astype(int)
    })

    submission.to_csv(output_path, index=False)
    print(f"Submission saved locally to: {output_path}")

    # UPLOAD CSV TO W&B ---
    print("Uploading Kaggle submission to W&B Cloud...")
    csv_artifact = wandb.Artifact(
        name=f"{file_name}_submission",
        type="predictions"
    )
    csv_artifact.add_file(output_path)
    wandb.log_artifact(csv_artifact)

    # --- 5. CLOSE THE W&B RUN ---
    wandb.finish()



#### Helper functions

In [4]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [5]:
def map_clusters_to_labels(y_true, y_clusters):
    """
    Unsupervised models assign arbitrary cluster IDs (e.g., 0-9).
    This function uses the Hungarian Algorithm to find the best mapping
    between cluster IDs and true labels to maximize accuracy.
    """
    # Ensure they are numpy arrays
    y_true = np.array(y_true).astype(int)
    y_clusters = np.array(y_clusters).astype(int)
    
    # Handle noise (-1) from DBSCAN/HDBSCAN by temporarily excluding it from mapping
    mask = y_clusters != -1
    
    # Create contingency matrix
    cm = confusion_matrix(y_true[mask], y_clusters[mask])
    
    # linear_sum_assignment minimizes cost, so we pass negative confusion matrix to maximize matches
    row_ind, col_ind = linear_sum_assignment(-cm)
    
    # Create a mapping dictionary
    mapping = {cluster_id: true_label for true_label, cluster_id in zip(row_ind, col_ind)}
    
    # Apply mapping
    y_mapped = np.copy(y_clusters)
    for i in range(len(y_clusters)):
        if y_clusters[i] != -1 and y_clusters[i] in mapping:
            y_mapped[i] = mapping[y_clusters[i]]
        else:
            y_mapped[i] = -1 # Keep noise as -1
            
    return y_mapped


# We will use Robust Scaled data for unsupervised models because distance metrics 
# (like Euclidean in KMeans or density in DBSCAN) are very sensitive to scale and outliers.
X_unsup = X_train_scaled_robust
y_unsup = y_train['position'].values

X_unsup_val = X_val_scaled_robust
y_unsup_val = y_val['position'].values

## Models

## K Means

In [6]:
print("\n--- Running K-Means ---")
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, init='k-means++', n_init=10, random_state=42)
# Train and predict on Validation
kmeans.fit(X_unsup)
kmeans_preds_val = kmeans.predict(X_unsup_val)

# Map clusters to actual positions
kmeans_mapped_preds = map_clusters_to_labels(y_unsup_val, kmeans_preds_val)

# Evaluate using your existing function
evaluate("KMeans", kmeans, X_unsup_val, y_unsup_val) 

metrics_kmeans = compute_metrics(y_unsup_val, kmeans_mapped_preds)
define_experiment("KMeans_Mapped", metrics_kmeans, y_unsup_val, kmeans_mapped_preds)


--- Running K-Means ---


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\laura\_netrc.


Evaluating best model on unseen validation data...
Initializing Weights & Biases run...


wandb: Currently logged in as: laura-rebollo-crespo (laura-rebollo-crespo-universitat-polit-cnica-de-catalunya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Initializing Weights & Biases run...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.21257
val_f1_macro,0.18824
val_precision_macro,0.196
val_recall_macro,0.21014


## Gaussian Mixture Models (GMM) / Expectation-Maximization (EM)

In [7]:
print("\n--- Running GMM (EM) ---")
from sklearn.mixture import GaussianMixture

# GMM is exactly Expectation-Maximization fitting a Gaussian Density Function
gmm = GaussianMixture(n_components=10, covariance_type='full', random_state=42)
gmm.fit(X_unsup)
gmm_preds_val = gmm.predict(X_unsup_val)

gmm_mapped_preds = map_clusters_to_labels(y_unsup_val, gmm_preds_val)
metrics_gmm = compute_metrics(y_unsup_val, gmm_mapped_preds)
define_experiment("GMM_EM_Mapped", metrics_gmm, y_unsup_val, gmm_mapped_preds)


--- Running GMM (EM) ---
Initializing Weights & Biases run...


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.50698
val_f1_macro,0.49233
val_precision_macro,0.54578
val_recall_macro,0.50381


## Fuzzy K-Means

In [8]:
%pip install skfuzzy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement skfuzzy (from versions: none)
ERROR: No matching distribution found for skfuzzy


In [12]:
import skfuzzy as fuzz

print("\n--- Running Fuzzy C-Means ---")
try:
    
    # skfuzzy requires data in shape (features, samples), so we transpose
    X_unsup_T = X_unsup.values.T
    X_unsup_val_T = X_unsup_val.values.T
    
    # Apply Fuzzy C-Means
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        X_unsup_T, c=10, m=2, error=0.005, maxiter=1000, init=None
    )
    
    # Predict for validation set
    u_val, u0_val, d_val, jm_val, p_val, fpc_val = fuzz.cluster.cmeans_predict(
        X_unsup_val_T, cntr, m=2, error=0.005, maxiter=1000
    )
    
    # Get crisp cluster assignments (highest probability)
    fuzzy_preds_val = np.argmax(u_val, axis=0)
    fuzzy_mapped_preds = map_clusters_to_labels(y_unsup_val, fuzzy_preds_val)
    
    metrics_fuzzy = compute_metrics(y_unsup_val, fuzzy_mapped_preds)
    define_experiment("Fuzzy_KMeans_Mapped", metrics_fuzzy, y_unsup_val, fuzzy_mapped_preds)

except ImportError:
    print("scikit-fuzzy not installed. Run `pip install scikit-fuzzy` to run Fuzzy K-Means.")


--- Running Fuzzy C-Means ---


c:\Users\laura\Desktop\UNI\Q4\AA1\ML-WiFi-Sensing-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\laura\Desktop\UNI\Q4\AA1\ML-WiFi-Sensing-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\laura\Desktop\UNI\Q4\AA1\ML-WiFi-Sensing-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter t

Initializing Weights & Biases run...


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.53995
val_f1_macro,0.52082
val_precision_macro,0.56406
val_recall_macro,0.53535


## Agglomerative Hierarchical Clustering

In [16]:
print("\n--- Running Agglomerative Clustering ---")
from sklearn.cluster import AgglomerativeClustering

agg_cluster = AgglomerativeClustering(n_clusters=10, metric='euclidean', linkage='ward')
agg_preds_val = agg_cluster.fit_predict(X_unsup_val)

agg_mapped_preds = map_clusters_to_labels(y_unsup_val, agg_preds_val)
metrics_agg = compute_metrics(y_unsup_val, agg_mapped_preds)
define_experiment("Agglomerative_Mapped", metrics_agg, y_unsup_val, agg_mapped_preds)


--- Running Agglomerative Clustering ---
Initializing Weights & Biases run...


## DBSCAN

In [13]:
from sklearn.decomposition import PCA

def log_density_clustering_wandb(base_name, y_true, y_clusters, X_data):
    """
    Registra mètriques de clustering no supervisat i un gràfic 2D a W&B.
    """
    ari = adjusted_rand_score(y_true, y_clusters)
    nmi = normalized_mutual_info_score(y_true, y_clusters)
    n_clusters = len(set(y_clusters)) - (1 if -1 in y_clusters else 0)
    n_noise = list(y_clusters).count(-1)
    
    print(f"Initializing W&B run for {base_name}...")
    wandb.init(
        project="AA1",
        entity="laura-rebollo-crespo-universitat-polit-cnica-de-catalunya",
        name=f"{base_name}_ARI-{ari:.4f}",
        config={
            "model_name": base_name, 
            "n_clusters_found": n_clusters, 
            "n_noise_points": n_noise
        }
    )
    
    # Crear un gràfic de PCA en 2D per visualitzar els clusters
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_data)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    # Utilitzem 'tab20' perquè hi pot haver molts clusters diferents
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_clusters, cmap='tab20', alpha=0.6, s=15)
    plt.title(f'{base_name} Clusters (PCA Projection)\nNoise points are colored dark/discrete', fontsize=14)
    plt.colorbar(scatter, label='Cluster ID (-1 = Noise)')
    plt.tight_layout()
    
    # Pujar mètriques i el gràfic a W&B
    wandb.log({
        "Adjusted_Rand_Index": ari,
        "Normalized_Mutual_Info": nmi,
        "Num_Clusters": n_clusters,
        "Num_Noise_Points": n_noise,
        "cluster_pca_scatter": wandb.Image(fig)
    })
    
    plt.close(fig)
    wandb.finish()
    print(f"Successfully logged {base_name} to W&B.\n")

In [17]:
print("\n--- Running DBSCAN ---")
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=5.0, min_samples=10) 
dbscan_preds_val = dbscan.fit_predict(X_unsup_val)

# Log to W&B using our new clustering function
log_density_clustering_wandb("DBSCAN", y_unsup_val, dbscan_preds_val, X_unsup_val)


--- Running DBSCAN ---
Initializing W&B run for DBSCAN...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.53995
val_f1_macro,0.52082
val_precision_macro,0.56406
val_recall_macro,0.53535


Adjusted_Rand_Index,▁
Normalized_Mutual_Info,▁
Num_Clusters,▁
Num_Noise_Points,▁
Adjusted_Rand_Index,0.11529
Normalized_Mutual_Info,0.23743
Num_Clusters,2
Num_Noise_Points,1277


Successfully logged DBSCAN to W&B.



## HDBSCAN (Hierarchical DBSCAN)

In [19]:
print("\n--- Running HDBSCAN ---")
try:
    from sklearn.cluster import HDBSCAN
    
    hdbscan = HDBSCAN(min_cluster_size=15, metric='euclidean')
    hdbscan_preds_val = hdbscan.fit_predict(X_unsup_val)
    
    # Log to W&B using our new clustering function
    log_density_clustering_wandb("HDBSCAN", y_unsup_val, hdbscan_preds_val, X_unsup_val)

except ImportError:
    print("HDBSCAN requires scikit-learn >= 1.3.0. Please update your environment.")


--- Running HDBSCAN ---
Initializing W&B run for HDBSCAN...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Adjusted_Rand_Index,▁
Normalized_Mutual_Info,▁
Num_Clusters,▁
Num_Noise_Points,▁
Adjusted_Rand_Index,0.03753
Normalized_Mutual_Info,0.15106
Num_Clusters,2
Num_Noise_Points,500


Successfully logged HDBSCAN to W&B.

